## Práctica 5: Clasificación de Emociones Faciales
Pipeline completo: extracción de características (**FaceNet**, **HOG**, **LBP**), entrenamiento/evaluación (**KNN** y **SVM**) y prueba en tiempo real con **Webcam**.

In [ ]:
import os
import numpy as np
import cv2

from time import time
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from deepface import DeepFace
from skimage.feature import hog
import joblib
from skimage import feature
from mtcnn.mtcnn import MTCNN
import collections


TRAIN = "dataset_emotions/train" 
TEST = "dataset_emotions/test"

### 1. Funciones de extracción, entrenamiento y evaluación

In [ ]:

def load_dataset(path, ext):
    X, Y = [], []
    clases = sorted(os.listdir(path))

    for idx,label in enumerate(clases):
        for file in os.listdir(os.path.join(path, label)):
            if file.endswith(ext):
                img = cv2.imread(os.path.join(path, label, file))
                
                embedding_objs = DeepFace.represent(img_path = img,model_name  = "Facenet",detector_backend="skip", enforce_detection = False)   
                img_embedding = embedding_objs[0]["embedding"]             

                X.append(img_embedding)
                Y.append(idx)
                
    X = np.array(X, dtype=np.float32)
    Y = np.array(Y)
    print(f"Features \n {X.shape} \n {Y.shape}")

    return X, Y, clases

def load_pixels(path, ext):
    X, Y = [], []
    clases = sorted(os.listdir(path))
    for idx, label in enumerate(clases):
        for file in os.listdir(os.path.join(path, label)):
            if file.endswith(ext):
                gray = cv2.imread(os.path.join(path, label, file), cv2.IMREAD_GRAYSCALE)
                X.append(gray.reshape(-1)) 
                Y.append(idx)
    return np.array(X, dtype=np.float32), np.array(Y), clases


def lbphist(gray, ncellsx, ncellsy, width, height, type, n_bins):
    #Rejilla de histogramas
    pxpercellx = int(width/ncellsx)
    pxpercelly = int(height/ncellsy)

    # Rejillas de igual tamaño y centradas en la imagen de entrada
    ofx = int((width - int(pxpercellx)*ncellsx)/2)
    ofy = int((height - int(pxpercelly)*ncellsy)/2)
   
    #Obtiene histograma LBP para cada celda
    LBPu_hist = []
    for i in range(0,ncellsy):
        for j in range(0,ncellsx):
            # Extrae celda
            roi=gray[ofy+i*pxpercelly:ofy+(i+1)*pxpercelly,ofx+j*pxpercellx:ofx+(j+1)*pxpercellx]
            # Obtiene imagen LBP
            lbpimg = feature.local_binary_pattern(roi, 8, 1,method=type)
                     
            #feath, bins = np.histogram(lbpimg.ravel(), normed=True, bins=n_bins, range=(0,n_bins))
            #feath, bins = np.histogram(lbpimg, normed=False, bins=n_bins, range=(0,n_bins)) #normed deprecated
            feath, bins = np.histogram(lbpimg, density=False, bins=n_bins, range=(0,n_bins))
            
            # Concatena con celdas previas
            LBPu_hist= np.concatenate([LBPu_hist , feath])
            
    return LBPu_hist

def GetHOG(X, ncellsx, ncellsy, width, height):
    Xhog = []
    for i in range(len(X)):
        #Recompone imagen
        gray = X[i].reshape(height, width)
        
        # HOG            
        feat_hog, hog_image = hog(gray, orientations=8, pixels_per_cell=(height/ncellsy, width/ncellsx),
                    cells_per_block=(1, 1), visualize=True, block_norm='L1')   #multichannel=False,  elimindo en versiones recientes
        Xhog.append(feat_hog)
            
    return Xhog
        
def GetLBP(X, ncellsx, ncellsy, width, height, tipo="nri_uniform"):
    # nri_uniform da 59 patrones posibles; uniform da 10
    n_bins = 59 if tipo == "nri_uniform" else 10
    Xlbp = []
    for i in range(len(X)):
        gray = X[i].reshape(height, width).astype(np.uint8)
        Xlbp.append(lbphist(gray, ncellsx, ncellsy, width, height, tipo, n_bins))
    return Xlbp

def GetSVMPredictions(X_train, X_test, y_train, y_test):
    
    print("+++++++++++++++++++++++++++++++++++++++++++\n")
    print("SVM Normalization...")
    scaler = MinMaxScaler()
    train_X = scaler.fit_transform(X_train)
    test_X = scaler.transform(X_test)

    print("SVM training...")
    t0 = time()
    parameters = {'C': [1, 10, 100],
              'gamma': ['scale', 0.01, 0.1]}
    # Grid serach across parameter range
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced'), parameters, cv=5
    )
    clf = clf.fit(train_X, y_train)
    print("done in %0.3fs" % (time() - t0))
    print("Best estimator found by grid search:")
    print(clf.best_estimator_)


    print("Predicting")
    t0 = time()
    y_pred = clf.predict(test_X)
    print("done in %0.3fs" % (time() - t0))
    
    return y_pred, y_test, clf, scaler


def GetKNNPredictions(k, X_train, X_test, y_train, y_test):
    
    t0 = time()
    # k = 5 
    model_px = KNeighborsClassifier(n_neighbors = k) 

    # fdtraining of model 
    model_px.fit(X_train, y_train) 
    print("done in %0.3fs" % (time() - t0))

    print("Predicting...")
    t0 = time()
    y_pred=model_px.predict(X_test)
    print("done in %0.3fs" % (time() - t0))
    
  
    return y_pred, y_test, model_px


### 2. Extracción y guardado de características


In [ ]:

print('Loading dataset (train)')
X_train, Y_train, clases = load_dataset(TRAIN,'.png')

print('Loading dataset (test)')
X_test, Y_test, clases = load_dataset(TEST,'.png')

print("Dataset info:")
print(f"Train -> Samples: {X_train.shape[0]} Features {X_train.shape[1]}")
print(f"Test -> Samples: {X_test.shape[0]} Features {X_test.shape[1]}")
print("Nº de clases:", len(clases))
print("Clases en el dataset:", clases)
print("Imágenes por clase (Train):", dict(zip(clases, np.bincount(Y_train))))
print("Imágenes por clase (Test):", dict(zip(clases, np.bincount(Y_test))))




Loading dataset (train)
Features 
 (15000, 128) 
 (15000,)
Loading dataset (test)
Features 
 (4000, 128) 
 (4000,)
Dataset info:
Train -> Samples: 15000 Features 128
Test -> Samples: 4000 Features 128
Nº de clases: 5
Clases en el dataset: ['angry', 'fearful', 'happy', 'sad', 'surprised']
Imágenes por clase (Train): {'angry': np.int64(3000), 'fearful': np.int64(3000), 'happy': np.int64(3000), 'sad': np.int64(3000), 'surprised': np.int64(3000)}
Imágenes por clase (Test): {'angry': np.int64(800), 'fearful': np.int64(800), 'happy': np.int64(800), 'sad': np.int64(800), 'surprised': np.int64(800)}


In [4]:
np.savez("embeddings.npz", X_train=X_train, Y_train=Y_train,
         X_test=X_test, Y_test=Y_test, clases=np.array(clases))

In [ ]:
datos = np.load("embeddings.npz")
X_train, Y_train = datos["X_train"], datos["Y_train"]
X_test, Y_test = datos["X_test"], datos["Y_test"]
clases = list(datos["clases"])
print(X_train.shape, X_test.shape, clases)

(15000, 128) (4000, 128) [np.str_('angry'), np.str_('fearful'), np.str_('happy'), np.str_('sad'), np.str_('surprised')]


### 3. Modelos base con FaceNet (KNN y SVM)

In [ ]:
# PASO 3 EVALUAR 

#Facenet+KNN
y_pred, y_test, knnModel = GetKNNPredictions(11, X_train, X_test,Y_train, Y_test)
print("\nFacenet+KNN Metrics")
print("Precisión %0.3f, Recall %0.3f" % (precision_score(y_test, y_pred, average='weighted'),recall_score(y_test, y_pred, average='weighted')))
print(classification_report(y_test, y_pred, target_names=clases))
print(confusion_matrix(y_test, y_pred, labels=range(len(clases))))      
    
#Facenet+SVM
y_pred, y_test, svm_model,svm_scaler = GetSVMPredictions(X_train, X_test,Y_train, Y_test)
print("\nFacenet+SVM Metrics")
print("Precisión %0.3f, Recall %0.3f" % (precision_score(y_test, y_pred, average='weighted'),recall_score(y_test, y_pred, average='weighted')))
print(classification_report(y_test, y_pred, target_names=clases))
print(confusion_matrix(y_test, y_pred, labels=range(len(clases))))    # Revisar lo de las clases   
    

# Modelos
joblib.dump({"model": knnModel, "clases": clases}, "modelo_emociones_knn.joblib")
joblib.dump({"model": svm_model, "scaler": svm_scaler, "clases": clases}, "modelo_emociones_svm.joblib")
print("Modelos guardados correctamente.")


done in 0.031s
Predicting...
done in 0.326s

Facenet+KNN Metrics
Precisión 0.440, Recall 0.423
              precision    recall  f1-score   support

       angry       0.41      0.50      0.45       800
     fearful       0.33      0.36      0.34       800
       happy       0.53      0.33      0.40       800
         sad       0.35      0.40      0.37       800
   surprised       0.59      0.53      0.56       800

    accuracy                           0.42      4000
   macro avg       0.44      0.42      0.43      4000
weighted avg       0.44      0.42      0.43      4000

[[399 140  53 145  63]
 [167 291  53 191  98]
 [150 152 260 158  80]
 [175 176  79 320  50]
 [ 94 134  50  98 424]]
+++++++++++++++++++++++++++++++++++++++++++

SVM Normalization...
SVM training...
done in 455.690s
Best estimator found by grid search:
SVC(C=10, class_weight='balanced')
Predicting
done in 3.833s

Facenet+SVM Metrics
Precisión 0.560, Recall 0.557
              precision    recall  f1-score   suppor

### 4. Descriptores HOG y Fusión Doble (FaceNet + HOG)

In [ ]:
P_train, Yp_train, _ = load_pixels(TRAIN, '.png')
P_test, Yp_test, _ = load_pixels(TEST, '.png')
print("Mismo orden en train:", np.array_equal(Yp_train, Y_train))
print("Mismo orden en test:", np.array_equal(Yp_test, Y_test))



H_train = np.array(GetHOG(P_train, 6, 6, 48, 48), dtype=np.float32)
H_test = np.array(GetHOG(P_test, 6, 6, 48, 48), dtype=np.float32)
print(H_train.shape)   # (15000, 288)

y_pred, _, svm_hog, scaler_hog = GetSVMPredictions(H_train, H_test, Y_train, Y_test)
print("\n===== HOG + SVM =====")
print(classification_report(Y_test, y_pred, target_names=clases))
print(confusion_matrix(Y_test, y_pred))


F_train = np.hstack([X_train, H_train])
F_test = np.hstack([X_test, H_test])

y_pred, _, svm_fus, scaler_fus = GetSVMPredictions(F_train, F_test, Y_train, Y_test)
print("\n===== FaceNet + HOG + SVM =====")
print(classification_report(Y_test, y_pred, target_names=clases))
print(confusion_matrix(Y_test, y_pred))

Mismo orden en train: True
Mismo orden en test: True
(15000, 288)
+++++++++++++++++++++++++++++++++++++++++++

SVM Normalization...
SVM training...
done in 1089.694s
Best estimator found by grid search:
SVC(C=10, class_weight='balanced')
Predicting
done in 18.802s

===== HOG + SVM =====
              precision    recall  f1-score   support

       angry       0.48      0.48      0.48       800
     fearful       0.46      0.41      0.44       800
       happy       0.66      0.68      0.67       800
         sad       0.43      0.47      0.45       800
   surprised       0.73      0.72      0.73       800

    accuracy                           0.55      4000
   macro avg       0.55      0.55      0.55      4000
weighted avg       0.55      0.55      0.55      4000

[[388 122  85 157  48]
 [127 331  67 188  87]
 [ 77  53 546  95  29]
 [162 132  89 373  44]
 [ 62  75  42  47 574]]
+++++++++++++++++++++++++++++++++++++++++++

SVM Normalization...
SVM training...
done in 1010.673s
Best es

In [ ]:

joblib.dump({
    "model": svm_fus.best_estimator_,
    "scaler": scaler_fus,
    "clases": clases,
    "hog_celdas": 6,        # rejilla de HOG usada (6x6)
    "tam": 48,              # tamaño de las imágenes del dataset
}, "modelo_emociones_fusion.joblib")
print("Guardado")

Guardado


### 5. Descriptores LBP y Fusión Triple (FaceNet + HOG + LBP)

In [17]:
P_train, Yp_train, _ = load_pixels(TRAIN, '.png')
P_test, Yp_test, _ = load_pixels(TEST, '.png')
print("Mismo orden en train:", np.array_equal(Yp_train, Y_train))
print("Mismo orden en test:", np.array_equal(Yp_test, Y_test))



H_train = np.array(GetHOG(P_train, 6, 6, 48, 48), dtype=np.float32)
H_test = np.array(GetHOG(P_test, 6, 6, 48, 48), dtype=np.float32)
print(H_train.shape)   # (15000, 288)


# ---------- LBP ----------
t0 = time()
L_train = np.array(GetLBP(P_train, 3, 3, 48, 48), dtype=np.float32)
L_test = np.array(GetLBP(P_test, 3, 3, 48, 48), dtype=np.float32)
print("LBP en %0.1fs" % (time() - t0), L_train.shape)   # (15000, 531) = 59 bins x 9 celdas

# Guardados para no recalcularlos nunca más
np.savez("descriptores.npz", H_train=H_train, H_test=H_test, L_train=L_train, L_test=L_test)

# LBP solo, para tener la cifra en la tabla
y_pred, _, svm_lbp, scaler_lbp = GetSVMPredictions(L_train, L_test, Y_train, Y_test)
print("\nLBP+SVM")
print(classification_report(Y_test, y_pred, target_names=clases))

# ---------- Los tres juntos ----------
T_train = np.hstack([X_train, H_train, L_train])   # 128 + 288 + 531 = 947
T_test = np.hstack([X_test, H_test, L_test])
print("Vector final:", T_train.shape[1], "caracteristicas")

y_pred, _, svm_tri, scaler_tri = GetSVMPredictions(T_train, T_test, Y_train, Y_test)
print("\nFaceNet+HOG+LBP+SVM")
print(classification_report(Y_test, y_pred, target_names=clases))
print(confusion_matrix(Y_test, y_pred, labels=range(len(clases))))

Mismo orden en train: True
Mismo orden en test: True
(15000, 288)
LBP en 12.5s (15000, 531)
+++++++++++++++++++++++++++++++++++++++++++

SVM Normalization...
SVM training...
done in 1631.636s
Best estimator found by grid search:
SVC(C=100, class_weight='balanced')
Predicting
done in 16.199s

LBP+SVM
              precision    recall  f1-score   support

       angry       0.46      0.48      0.47       800
     fearful       0.43      0.39      0.41       800
       happy       0.63      0.65      0.64       800
         sad       0.43      0.43      0.43       800
   surprised       0.70      0.70      0.70       800

    accuracy                           0.53      4000
   macro avg       0.53      0.53      0.53      4000
weighted avg       0.53      0.53      0.53      4000

Vector final: 947 caracteristicas
+++++++++++++++++++++++++++++++++++++++++++

SVM Normalization...
SVM training...
done in 2560.669s
Best estimator found by grid search:
SVC(C=10, class_weight='balanced', gamm

In [ ]:


joblib.dump({
    "model": svm_tri.best_estimator_,
    "scaler": scaler_tri,
    "clases": clases,
    "tam": 48,
    "hog_celdas": 6,
    "lbp_celdas": 3,
    "lbp_tipo": "nri_uniform",
}, "modelo_emociones_tri.joblib")
print("Guardado:", svm_tri.best_estimator_)

Guardado: SVC(C=10, class_weight='balanced', gamma=0.1)


### 6. Prueba en tiempo real con Webcam

In [12]:
datos = joblib.load("modelo_emociones_tri.joblib")
svm, scaler, clases = datos["model"], datos["scaler"], [str(c) for c in datos["clases"]]
detector = MTCNN()
historial = collections.deque(maxlen=8)

def caracteristicas(cara):
    gris = cv2.resize(cv2.cvtColor(cara, cv2.COLOR_BGR2GRAY), (48, 48), interpolation=cv2.INTER_AREA)

    # FaceNet (128)
    emb = DeepFace.represent(img_path=cv2.cvtColor(gris, cv2.COLOR_GRAY2BGR), model_name="Facenet",
                             detector_backend="skip", enforce_detection=False)[0]["embedding"]
    # HOG 6x6 (288)
    h = hog(gris.astype(np.float32), orientations=8, pixels_per_cell=(8, 8),
            cells_per_block=(1, 1), block_norm='L1')
    
    # LBP 3x3 (531): 59 valores por celda
    l = []
    for i in range(3):
        for j in range(3):
            roi = gris[i * 16:(i + 1) * 16, j * 16:(j + 1) * 16]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method="nri_uniform")
            l.append(np.histogram(lbpimg, bins=59, range=(0, 59))[0])

    return np.hstack([emb, h, np.concatenate(l)]).reshape(1, -1)

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
t, emocion = 0, ""


while True:
    ret, frame = cap.read()
    if not ret:
        break
    t += 1
    results = detector.detect_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if results:
        for result in results:
            x, y, width, height = result['box']
            cv2.rectangle(frame, (x, y), (x + width, y + height), (0, 0, 255), 2)

            if t % 3 == 0:   # 1 de cada 3 fotogramas, para que no vaya a pedales
                historial.append(clases[int(svm.predict(scaler.transform(caracteristicas(frame[y:(y + height), x:(x + width)])))[0])])
            if historial:
                emocion = collections.Counter(historial).most_common(1)[0][0]

            cv2.putText(frame, emocion, (x, max(25, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
    cv2.imshow("Prueba", frame)
    if cv2.waitKey(1) == 27:
        break


cap.release()
cv2.destroyAllWindows()
